# LUMIERE run r1: what the 5 mask exclusions are (read-only check)

EXP-2026-005, follow-up to run r1. Run r1 read every HD-GLIO-AUTO file one way (flipped, 30 of 33 votes). This notebook measures, for the pairs r1 excluded and for the studies that voted otherwise, how well the two tools agree under each way of reading the files.

- Reads run r1's records from `MyDrive/LUMIERE_derived/EXP-2026-005/r1` through the Drive API, checked by MD5 and by their recorded SHA-256. Reads LUMIERE through the mount.
- Writes nothing to Drive and changes nothing in r1.
- Prints counts and lettered rows only, without patient names.
- No GPU needed: a CPU runtime is enough. I expect about 5 minutes (not measured).

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-lumiere"
LUMIERE_ROOT = "/content/drive/MyDrive/LUMIERE"
R1_RUN = ("LUMIERE_derived", "EXP-2026-005", "r1")  # run r1's folder under MyDrive, read only
LOCAL_COPY = "/content/lumiere_exp005_r1_records"  # r1's records copied to this runtime; they hold patient names

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
assert Path(LUMIERE_ROOT).is_dir(), f"{LUMIERE_ROOT} not found: check the folder name"
if not Path(WORK).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "SimpleITK==2.5.6", "nibabel>=5.3,<6"], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

from google.colab import auth
from googleapiclient.discovery import build

from rhuh_synthesis.drive_api import DriveRun

auth.authenticate_user()  # Colab asks once for permission to use the Drive API with this account
r1 = DriveRun.open(build("drive", "v3", cache_discovery=False), (), R1_RUN, LOCAL_COPY, create=False)
for name in ("run.json", "manifest.json", "mask_check.json"):
    for suffix in ("", ".sha256"):
        r1.fetch(f"outputs/{name}{suffix}")  # downloaded by folder ID, checked against the server's MD5
print("run r1 records fetched")

In [ ]:
from rhuh_synthesis import lumiere_reading_check as C

records = Path(LOCAL_COPY) / "outputs"
run, _ = C.load_record(records / "run.json")
manifest, manifest_sha = C.load_record(records / "manifest.json")
mask, _ = C.load_record(records / "mask_check.json")
assert manifest_sha == run["manifest_sha256"], "manifest.json is not the one run r1 recorded: stop and report"
started = time.time()


def say(message):
    print(f"[{time.time() - started:6.0f} s] {message}", flush=True)


result = C.reading_diagnostic(LUMIERE_ROOT, manifest, mask, progress=say)
print("DIAGNOSTIC")
print(json.dumps(result, indent=1))

## What to send back

Copy everything printed after `DIAGNOSTIC`. It holds counts and rows labelled A, B, C and so on, never patient names.

If a cell stops with an error, send the error message instead.